## Artificial Neural Networks

In [9]:
import numpy as np
import time 

Forward propagation

In [10]:

def sigmoid(z):
    # sigmoid activation function
    return 1 / (1 + np.exp(-z))

# the 8 examples
X = np.eye(8)
Y = X.copy()     #target 

# random starting weights and zero biases 
rng = np.random.default_rng(0)
W1 = rng.normal(0, 0.5, (8, 3))   # input (8) -> hidden (3)
b1 = np.zeros(3)
W2 = rng.normal(0, 0.5, (3, 8))   # hidden (3) -> output (8)
b2 = np.zeros(8)

# Forward propagation 
A1 = sigmoid(X @ W1 + b1)    # hidden values
A2 = sigmoid(A1 @ W2 + b2)   # output guesses

np.set_printoptions(precision=2, suppress=True)
print("Hidden values (8 examples x 3 hidden neurons):")
print(A1)
print("\nOutput guesses (8 examples x 8 outputs):")
print(A2)
m = X.shape[0]                              
J = 0.5 * np.sum((A2 - Y) ** 2) / m
print("\nCost:", J)

Hidden values (8 examples x 3 hidden neurons):
[[0.52 0.48 0.58]
 [0.51 0.43 0.55]
 [0.66 0.62 0.41]
 [0.35 0.42 0.51]
 [0.24 0.47 0.35]
 [0.41 0.43 0.46]
 [0.55 0.63 0.48]
 [0.66 0.42 0.54]]

Output guesses (8 examples x 8 outputs):
[[0.46 0.64 0.56 0.52 0.45 0.48 0.59 0.71]
 [0.46 0.64 0.56 0.51 0.45 0.49 0.58 0.69]
 [0.5  0.62 0.52 0.49 0.43 0.49 0.55 0.69]
 [0.45 0.63 0.56 0.53 0.46 0.48 0.59 0.68]
 [0.46 0.6  0.55 0.53 0.46 0.49 0.58 0.66]
 [0.47 0.62 0.55 0.52 0.46 0.49 0.57 0.68]
 [0.47 0.64 0.55 0.51 0.43 0.49 0.58 0.71]
 [0.48 0.64 0.54 0.5  0.45 0.49 0.56 0.68]]

Cost: 1.1654551995739066


Backpropagation

In [11]:

lr = 2.0         # learning rate
# delta of the output 
d_out = (A2 - Y) * A2 * (1 - A2)

# delta of the hidden 
d_hid = (d_out @ W2.T) * A1 * (1 - A1)

# update every weight
W2 -= lr * (A1.T @ d_out) / m
b2 -= lr * d_out.mean(axis=0)
W1 -= lr * (X.T @ d_hid) / m
b1 -= lr * d_hid.mean(axis=0)

# forward propagation with the new weights and check the cost
A1 = sigmoid(X @ W1 + b1)
A2 = sigmoid(A1 @ W2 + b2)
J = 0.5 * np.sum((A2 - Y) ** 2) / m
print("J after:", J)

J after: 0.8929764552310329


Training loop

In [12]:
for epoch in range(1, 20001):
    # forward
    A1 = sigmoid(X @ W1 + b1)
    A2 = sigmoid(A1 @ W2 + b2)
    J = 0.5 * np.sum((A2 - Y) ** 2) / m

    if epoch in (1, 100, 1000, 5000, 10000, 20000):
        print(f"epoch {epoch:>5}   J = {J:.4f}")

    # backward
    d_out = (A2 - Y) * A2 * (1 - A2)
    d_hid = (d_out @ W2.T) * A1 * (1 - A1)
    W2 -= lr * (A1.T @ d_out) / m
    b2 -= lr * d_out.mean(axis=0)
    W1 -= lr * (X.T @ d_hid) / m
    b1 -= lr * d_hid.mean(axis=0)

epoch     1   J = 0.8930
epoch   100   J = 0.4218
epoch  1000   J = 0.0622
epoch  5000   J = 0.0056
epoch 10000   J = 0.0025
epoch 20000   J = 0.0012


Training Function to run multiple experiments

In [13]:
def train_network(lr=2.0, seed=0, init_scale=0.5, max_epochs=20000, tol=0.01):
    # random starting weights every time
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, init_scale, (8, 3)); b1 = np.zeros(3)
    W2 = rng.normal(0, init_scale, (3, 8)); b2 = np.zeros(8)
    m = X.shape[0]
    history, converged_at = [], None

    for epoch in range(1, max_epochs + 1):
        # forward
        A1 = sigmoid(X @ W1 + b1)
        A2 = sigmoid(A1 @ W2 + b2)
        J = 0.5 * np.sum((A2 - Y) ** 2) / m
        history.append(J)
        # converged = first epoch where all guesses are correct and cost is below tolerance
        if converged_at is None and np.all(A2.argmax(1) == Y.argmax(1)) and J < tol:
            converged_at = epoch
        # backward
        d_out = (A2 - Y) * A2 * (1 - A2)
        d_hid = (d_out @ W2.T) * A1 * (1 - A1)
        W2 -= lr * (A1.T @ d_out) / m;  b2 -= lr * d_out.mean(axis=0)
        W1 -= lr * (X.T @ d_hid) / m;   b1 -= lr * d_hid.mean(axis=0)

    A1 = sigmoid(X @ W1 + b1)   # final hidden values
    return converged_at, history, A1

In [14]:

start = time.time()
converged_at, history, A1 = train_network(lr=2.0, seed=0)

print("Converged at epoch:", converged_at)
print("Time:", round(time.time() - start, 2), "seconds")
print("Final J:", history[-1])
print("\nHidden code (rounded):\n", np.round(A1).astype(int))

Converged at epoch: 3131
Time: 1.34 seconds
Final J: 0.0011739472915837468

Hidden code (rounded):
 [[1 0 0]
 [1 0 1]
 [1 1 0]
 [0 0 1]
 [0 1 0]
 [0 0 0]
 [0 1 1]
 [1 1 1]]
